In [8]:
import csv
import importlib
import os
import random
import sys
import torch
import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

import ray
from ray import tune
from ray.tune.schedulers.pb2 import PB2, PopulationBasedTraining
from ray.tune import Checkpoint, run, sample_from 

from time import sleep
from collections import deque, defaultdict
from itertools import count
from typing import Any, Dict, Counter, List

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
SOURCES_ROOT = os.path.join(PROJECT_ROOT, 'Sources')
RAY_EXCLUDES = [
    '.git/',
    '.venv/',
    'Dataset/',
    'Results/',
    'SampleVideos/',
    'Sources/Tests/deepmimo_scenarios/',
    'Sources/Tests/deepmimo_scenarios/*.zip',
]

for path in [PROJECT_ROOT, SOURCES_ROOT]:
    if path not in sys.path:
        sys.path.insert(0, path)

from importnb import Notebook

from RL.Adapters import FeatureAdapter, NetworkAdapter
from RL.Buffers import ReplayBuffer, NStepReplayBuffer
from RL.Networks import QNetwork, MultiHeadQNetwork
from RL.Models.Focus import Focus

import Common.config as config
import Common.datatypes as datatypes
import Common.debugger as debugger
import Common.utils as utils

importlib.reload(config)
importlib.reload(datatypes)
importlib.reload(debugger)
importlib.reload(utils)


<module 'Common.utils' from '/home/eduardo/Workspace/CacheVideoPredict360/Sources/Common/utils.py'>

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CachePolicy = datatypes.CachePolicy
CacheKey = datatypes.CacheKey

cfg = config.Config()

debugger = debugger.debug

In [10]:
class DrlPolicy(CachePolicy):
    def __init__(self, cfg: Any = None):
        self.cfg = cfg
        self.cur_size = 0

        self.video_idx = [-1] * self.cfg.cache_size
        self.tile_idx = [[-1] * self.cfg.viewport for _ in range(self.cfg.cache_size)]

    def get(self, key: CacheKey) -> Any:
        return self.cache.get(key, None)

    def put(self, key: int, value: Any, size: int) -> list:
        """
        key   -> slot index
        value -> (video_id, tiles)
        size  -> fixed as 1 slot
        """
        evicted = []
        slot = key
        new_video, _ = value

        if new_video in self.video_idx:
            old_video_slot = self.video_idx.index(new_video)
            if old_video_slot != slot:
                # Swap video positions
                self.video_idx[slot], self.video_idx[old_video_slot] = (
                    self.video_idx[old_video_slot],
                    self.video_idx[slot],
                )

                # Swap tile mapping accordingly
                self.tile_idx[slot], self.tile_idx[old_video_slot] = (
                    self.tile_idx[old_video_slot],
                    self.tile_idx[slot],
                )

            self.cur_size = sum(1 for v in self.video_idx if v != -1)
            return evicted

        self.video_idx[slot] = new_video
        self.tile_idx[slot] = [-1] * self.cfg.viewport

        self.cur_size = sum(1 for v in self.video_idx if v != -1)
        return evicted

    def contains(self, key: CacheKey) -> bool:
        return key in self.video_idx

    def remove(self, key: CacheKey) -> bool:
        if key in self.video_idx:
            idx = self.video_idx.index(key)
            self.video_idx[idx] = -1
            self.tile_idx[idx] = [-1] * self.cfg.viewport
            self.cur_size = sum(1 for v in self.video_idx if v != -1)
            return True
        return False

    def clear(self) -> None:
        self.video_idx = [-1] * self.cfg.cache_size
        self.tile_idx = [[-1] * self.cfg.viewport for _ in range(self.cfg.cache_size)]
        self.cur_size = 0

    def keys(self):
        return self.video_idx
    
    def get_capacity(self) -> int:
        return self.cur_size
    
    def update_size(self):
        self.cur_size = sum(1 for v in self.video_idx if v != -1)

    def stats(self) -> Dict[str, Any]:
        return {
            'current_size': self.cur_size,
            'capacity': self.cfg.cache_size,
            'num_items': len([v for v in self.video_idx if v != -1])
        }

In [11]:
def build_latency_model(cfg):
    """Build and return the MultiDULatencyModel."""
    from importnb import Notebook
    with Notebook():
        from Labs.LatencyModel import MultiDULatencyModel

    P = cfg.n_nodes
    max_U = cfg.n_users

    return MultiDULatencyModel(
        P=P,
        max_U=max_U,
        R_M_D=80e6,
        R_C_M=125e6,
        mu=2e7,
        eta=2e5,
        B_pu_matrix=np.full((P, max_U), 20e6, dtype=float),
        gamma_pu_matrix=np.full((P, max_U), 5.0, dtype=float),
        rhoT_p=[0.2],
        lambda_p=[0.05],
        du_fixed_delay=0.001,
        mec_fixed_delay=0.005,
        cloud_fixed_delay=0.1
    )

def build_environment(cfg):
    """Construct the full multi-component environment wrapper."""
    from importnb import Notebook
    with Notebook():
        from Labs.CacheEngine import CacheEngineEnv
        from Labs.UserRequest import UserRequestEvents
        from Labs.EnvWrapperFocus import EnvWrapper

    du_caches = []

    # DRL Caching Policy
    policy = DrlPolicy(cfg=cfg)

    # MEC Cache Engine
    mec_cache = CacheEngineEnv(
        n_users=cfg.n_users,
        n_videos=cfg.n_videos,
        n_layers=cfg.n_layers,
        n_tiles=cfg.n_tiles,
        n_gops=cfg.n_gops,
        cache_capacity=cfg.cache_capacity,
        policy=policy
    )

    # User request generator
    users_env = UserRequestEvents(
        n_nodes=cfg.n_nodes,
        n_users=cfg.n_users,
        n_videos=cfg.n_videos,
        n_gops=cfg.n_gops,
        n_layers=cfg.n_layers,
        n_tiles=cfg.n_tiles,
        n=cfg.n,
        m=cfg.m,
        arrival_rate=cfg.arrival_rate,
        zipf_alpha=cfg.zipf_alpha
    )

    # Latency Model
    latency_model = build_latency_model(cfg)

    # Wrapping all into the main training environment
    return EnvWrapper(
        cfg=cfg,
        n=cfg.n,
        m=cfg.m,
        n_layers=cfg.n_layers,
        users_env=users_env,
        du_caches=du_caches,
        mec_cache=mec_cache,
        latency_model=latency_model,
        theta=cfg.theta,
        lam=cfg.lam,
        max_steps=cfg.max_steps,
        prefetch_fn=lambda cache, action: cache.drl_prefetching_pantelis(action),
        reward_fn=lambda env, reqs: env.compute_reward(reqs),
        debugger=debugger
    )

In [12]:
class PGTrainer(object):
    def __init__(
        self, 
        env, 
        model, 
        optimizer, 
        scheduler, 
        gamma=0.99, 
        update_target_every=10
    ):
        self.env = env
        self.model = model
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.gamma = gamma
        self.update_target_every = update_target_every
        self.n_step_buffer = []

In [ ]:
def rl(config):
    
    print(config)
        
    env = build_environment(cfg)
    model = Focus(cfg)

    trainer = PGTrainer(
        env=env,
        model=model,
        optimizer=optim.Adam,
        scheduler=None,
        gamma=cfg.gamma,
        update_target_every=10
    )

    start_episode = 0
    checkpoint = tune.get_checkpoint()
    if checkpoint:
        with checkpoint.as_directory() as checkpoint_dir:
            checkpoint_data = torch.load(
                os.path.join(checkpoint_dir, "checkpoint.pt"),
                map_location=device
            )

            start_episode = checkpoint_data["episode"] + 1
            trainer.behaviour_net.load_state_dict(checkpoint_data["model_state_dict"])
            print(f"== resume from checkpoint, continue with epoch {start_episode} \n")

        rewards = []

        for episode in range(cfg.n_episodes):
            reward = trainer.run(episode)
            rewards.append(reward)

            if episode % cfg.nb_interval == 0:
                checkpoint_data = {
                    "episode": episode,
                    "model_state_dict": trainer.behaviour_net.state_dict(),
                }
                with tune.checkpoint_dir(episode) as checkpoint_dir:
                    torch.save(checkpoint_data, os.path.join(checkpoint_dir, "checkpoint.pt"))

    for i in range(cfg.n_episodes):
        stat = {}
        hyperparams = {
            "n_step": cfg.n_step,
            "gamma": cfg.gamma,
            "lr": cfg.learning_rate,
            "batch_size": cfg.batch_size,
            "buffer_capacity": cfg.buffer_capacity,
        }
        trainer.run(stat, i, hyperparams=hyperparams)

In [14]:
if __name__ == "__main__":
    if ray.is_initialized():
        ray.shutdown()

    ray.init(
        ignore_reinit_error=True,
        runtime_env={
            "working_dir": PROJECT_ROOT,
            "excludes": RAY_EXCLUDES,
            "env_vars": {
                "PYTHONPATH": f"{PROJECT_ROOT}:{SOURCES_ROOT}",
            },
        },
    )

    pb2 = PB2(
        metric="objective",
        mode="max",
        quantile_fraction=0.25,
        perturbation_interval=160,
        hyperparam_bounds={
            "alpha": [0.05, 0.2],
            "beta": [0.3, 0.6]
        }
    )

    print("== Starting Ray Tune with Population Based Training (PB2) scheduler ==")

    for seed in range(0, 4):
        analysis = tune.run(
            rl,
            scheduler=pb2,
            num_samples=1,
            reuse_actors=True,
            config={
                "alpha": sample_from(lambda spec: np.random.uniform(0.05, 0.2)),
                "beta": sample_from(lambda spec: np.random.uniform(0.4, 0.6)),
                "seed": seed
            }
        )

        all_dfs = analysis.trial_dataframes
        names = list(all_dfs.keys())

        results = pd.DataFrame()
        for i in range(4):
            df = all_dfs[names[i]].copy()
            df['sample_num'] = i 
            results = pd.concat([results, df]).reset_index(drop=True)

        dir = "{}_{}_{}_Size{}_{}_{}_{}_{}_{}".format(rl, "file", "method", str(4), "env", "default", "max", "160", "batch")
        exist_dir = os.path.expanduser('~/data/' + dir)
        if not(os.path.exists(exist_dir)):
            os.makedirs(exist_dir)

        result_dir1 = os.path.expanduser('~/data/')
        result_dir2 = f"{dir}/seed{seed}.csv"
        results.to_csv(result_dir1 + result_dir2)

2026-02-26 23:46:42,068	INFO worker.py:2013 -- Started a local Ray instance.
2026-02-26 23:46:42,076	INFO packaging.py:392 -- Ignoring upload to cluster for these files: [PosixPath('/home/eduardo/Workspace/CacheVideoPredict360/.gitignore')]
2026-02-26 23:46:42,108	INFO packaging.py:392 -- Ignoring upload to cluster for these files: [PosixPath('/home/eduardo/Workspace/CacheVideoPredict360/.venv/.gitignore')]
2026-02-26 23:46:42,110	INFO packaging.py:691 -- Creating a file package for local module '/home/eduardo/Workspace/CacheVideoPredict360'.
2026-02-26 23:46:42,111	INFO packaging.py:392 -- Ignoring upload to cluster for these files: [PosixPath('/home/eduardo/Workspace/CacheVideoPredict360/.gitignore')]
2026-02-26 23:46:42,139	INFO packaging.py:392 -- Ignoring upload to cluster for these files: [PosixPath('/home/eduardo/Workspace/CacheVideoPredict360/.venv/.gitignore')]
2026-02-26 23:46:42,148	INFO packaging.py:463 -- Pushing file package 'gcs://_ray_pkg_dc915625d98b6131.zip' (18.43MiB

== Starting Ray Tune with Population Based Training (PB2) scheduler ==


(rl pid=97527) {'alpha': 0.09495749173943163, 'beta': 0.5141744069431371, 'seed': 0}


2026-02-26 23:46:45,638	ERROR tune_controller.py:1331 -- Trial task failed for trial rl_67115_00000
Traceback (most recent call last):
  File "/home/eduardo/Workspace/CacheVideoPredict360/.venv/lib/python3.12/site-packages/ray/air/execution/_internal/event_manager.py", line 110, in resolve_future
    result = ray.get(future)
             ^^^^^^^^^^^^^^^
  File "/home/eduardo/Workspace/CacheVideoPredict360/.venv/lib/python3.12/site-packages/ray/_private/auto_init_hook.py", line 22, in auto_init_wrapper
    return fn(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^
  File "/home/eduardo/Workspace/CacheVideoPredict360/.venv/lib/python3.12/site-packages/ray/_private/client_mode_hook.py", line 104, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/home/eduardo/Workspace/CacheVideoPredict360/.venv/lib/python3.12/site-packages/ray/_private/worker.py", line 2981, in get
    values, debugger_breakpoint = worker.get_objects(
                                  ^^

Trial name
rl_67115_00000


2026-02-26 23:46:45,644	INFO tune.py:1009 -- Wrote the latest version of all result files and experiment state to '/home/eduardo/ray_results/rl_2026-02-26_23-46-42' in 0.0016s.


TuneError: ('Trials did not complete', [rl_67115_00000])